# 11 - REST vs GraphQL vs gRPC

## 1. REST
**Definition:** Each resource has its own URL, and HTTP methods (GET, POST, PUT, DELETE) act on it. Data comes back as JSON.

**Limitation:**
- **Over-fetching:** one call returns more fields than you need
- **Under-fetching:** you need several calls to get related data

**Real-world example:** Razorpay's payments API, where your backend calls POST /payments to create a payment.

## 2. GraphQL
**Definition:** A single endpoint (/graphql) where the client sends a query listing exactly the fields it needs. It can include related data in the same request.

**Limitation:** Harder to cache, more complex to set up, and a deeply nested query can put a heavy load on the server.

**Real-world example:** A mobile app's product page asks for just the name, price and top 3 reviews in one request, instead of three REST calls.

## 3. gRPC
**Definition:** A fast way for services to call each other. Data is sent as compact binary **Protocol Buffers** over **HTTP/2**, with a strict contract defined in a .proto file.

**Limitation:** Messages aren't human-readable, browsers don't support it well, and debugging is harder than with JSON.

**Real-world example:** At checkout, the Order service calls the Payment and Inventory services internally thousands of times a second.

## Comparison

| | REST | GraphQL | gRPC |
|---|---|---|---|
| Endpoints | Many URLs | One | Service methods |
| Format | JSON | JSON | Binary (Protobuf) |
| Best for | Public APIs, web apps | Mobile, complex UIs | Internal microservices |
| Main weakness | Over/under-fetching | Caching, complexity | Not human-readable |

**Interview line:** "REST for public APIs, GraphQL when clients need flexible data, and gRPC for fast internal calls between microservices. My projects use REST with FastAPI."

In [ ]:
# ==================================================
# 1. OVER-FETCHING (REST) vs FIELD SELECTION (GraphQL)
# ==================================================
import json, struct

product = {"id": 5, "name": "Laptop", "price": 55000, "description": "16GB RAM, 512GB SSD...",
           "stock": 12, "seller": "TechStore", "images": ["a.jpg", "b.jpg"], "rating": 4.5}
reviews = [{"user": "A", "stars": 5}, {"user": "B", "stars": 4}, {"user": "C", "stars": 5}, {"user": "D", "stars": 3}]

# REST: fixed shape, 2 calls
rest_calls = [product, reviews]
print("=" * 60)
print(f"[REST]    2 calls, {len(json.dumps(rest_calls))} bytes, all fields returned")

# GraphQL: client picks fields, 1 call
query_fields = ["name", "price"]
graphql_result = {k: product[k] for k in query_fields}
graphql_result["reviews"] = reviews[:3]
print(f"[GraphQL] 1 call,  {len(json.dumps(graphql_result))} bytes -> {graphql_result}")

# ==================================================
# 2. JSON vs BINARY SIZE (why gRPC is fast)
# ==================================================
order = {"order_id": 1001, "qty": 3, "amount": 55000.0}
json_bytes = json.dumps(order).encode()
binary_bytes = struct.pack("iid", order["order_id"], order["qty"], order["amount"])  # Protobuf-like idea

print(f"[JSON]    {len(json_bytes)} bytes -> {json_bytes}")
print(f"[BINARY]  {len(binary_bytes)} bytes -> not human-readable")
print("=" * 60)

## Key Takeaways

- REST: many URLs, JSON, simple; suffers over- and under-fetching
- GraphQL: one endpoint, client picks fields; harder to cache
- gRPC: binary Protobuf over HTTP/2, fastest; not readable, poor browser support
- Public → REST, flexible clients → GraphQL, internal services → gRPC